# Project 12 — Cat vs Dog Classifier (Transfer Learning)

This notebook walks through the full pipeline:
1. Download & explore the Cats vs Dogs dataset
2. Build a MobileNetV2-based classifier with a frozen backbone
3. Phase 1: train the classification head
4. Phase 2: fine-tune the top backbone layers
5. Evaluate and plot the training curve
6. Save the model
7. Run inference on custom images

It reuses the exact same code as `src/data.py`, `src/model.py`, and `src/train.py` so results here match what the scripts produce.

In [ ]:
import sys
from pathlib import Path

# Allow imports from the project's src/ package when running from notebooks/
ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import (
    MODEL_PATH, TRAINING_CURVE_PATH, CUSTOM_IMAGES_DIR,
    HEAD_EPOCHS, FINE_TUNE_EPOCHS, IMG_SIZE
)
from src.data import load_datasets
from src.model import build_model, enable_fine_tuning
from src.predict import preprocess_image_path

print("TensorFlow version:", tf.__version__)

## 1. Download and explore the data

This downloads the ~68MB Cats vs Dogs dataset on first run (cached afterward under `data/raw/`).

In [ ]:
train_ds, val_ds, class_names = load_datasets()
print("Class names:", class_names)

In [ ]:
# Visualize a batch of training images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.suptitle("Sample training images")
plt.show()

## 2. Build the model

MobileNetV2 backbone (frozen) + `GlobalAveragePooling2D` + `Dropout(0.2)` + `Dense(128, relu)` + `Dense(1, sigmoid)`. MobileNetV2's `preprocess_input` is baked directly into the model, so raw 0-255 RGB images can be fed in without manual normalization anywhere (script, notebook, or app).

In [ ]:
model, base_model = build_model(weights="imagenet")
model.summary()

## 3. Phase 1 — train the classification head (backbone frozen)

In [ ]:
history1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS)

## 4. Phase 2 — fine-tune the top backbone layers

Unfreezes the last layers of MobileNetV2 and recompiles with a much lower learning rate (1e-5) so we adapt the pretrained features without destroying them.

In [ ]:
enable_fine_tuning(model, base_model)
history2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS)

## 5. Evaluate and plot the combined training curve

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Final validation accuracy: {val_acc:.4f}")
print(f"Final validation loss: {val_loss:.4f}")

In [ ]:
acc = history1.history["accuracy"] + history2.history["accuracy"]
val_accuracy = history1.history["val_accuracy"] + history2.history["val_accuracy"]

plt.figure(figsize=(7, 5))
plt.plot(acc, label="train_acc")
plt.plot(val_accuracy, label="val_acc")
plt.axvline(x=HEAD_EPOCHS - 0.5, color="gray", linestyle="--", label="fine-tuning starts")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")
plt.title("Training Curve — Cat vs Dog Transfer Learning")
plt.tight_layout()
plt.savefig(TRAINING_CURVE_PATH, dpi=150)
plt.show()
print(f"Saved training curve to {TRAINING_CURVE_PATH}")

## 6. Save the trained model

In [ ]:
MODEL_PATH.parent.mkdir(exist_ok=True, parents=True)
model.save(MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")

## 7. Run inference on custom images

Add your own `.png`/`.jpg` cat or dog photos to `data/custom_images/` and re-run this cell. This reuses `src/predict.py`'s `preprocess_image_path`, the same function used by the standalone CLI script and the Streamlit app.

In [ ]:
image_paths = sorted(
    p for ext in ("*.png", "*.jpg", "*.jpeg") for p in CUSTOM_IMAGES_DIR.glob(ext)
)

if not image_paths:
    print(f"No custom images found in {CUSTOM_IMAGES_DIR}.")
    print("Add some cat/dog photos (.png/.jpg) there and re-run this cell.")
else:
    n = len(image_paths)
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
    axes = np.array(axes).flatten() if n > 1 else [axes]

    for ax, img_path in zip(axes, image_paths):
        img_array = preprocess_image_path(img_path)
        prob = float(model.predict(img_array, verbose=0)[0][0])
        label = class_names[1] if prob >= 0.5 else class_names[0]
        confidence = prob if prob >= 0.5 else 1 - prob
        print(f"{img_path.name}: predicted={label} confidence={confidence:.2%}")

        ax.imshow(img_array[0].astype("uint8"))
        ax.set_title(f"{img_path.name}\nPred: {label} ({confidence:.0%})")
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Summary

- Built a MobileNetV2-based transfer learning model for binary cat/dog classification
- Phase 1 trained only the classification head (fast, backbone frozen)
- Phase 2 fine-tuned the top backbone layers at a low learning rate for a further accuracy boost
- Saved the model to `models/cat_dog_transfer.h5` and the training curve to `reports/figures/training_curve.png`
- Verified inference works on custom uploaded images
- The same model powers `src/predict.py` (CLI) and `app/streamlit_app.py` (web UI)

**Key insight:** transfer learning lets a small dataset reach strong accuracy quickly by reusing features already learned from ImageNet — freezing the backbone first avoids destroying those features early on, and the later low-learning-rate fine-tuning squeezes out extra accuracy without overfitting.